In [1]:
import pandas as pd, numpy as np, statsmodels.api as sm

df = pd.read_excel("../data/sp500_event_study_data.xlsx")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)
df["Y_t"] = df["Close"].pct_change() * 100
df["Dt"] = df["Dt"].fillna(0).astype(int)
df.loc[df["Date"] == "2026-01-26", "Dt"] = 1            # ngày sự kiện thứ 5
df["Y_t_lag1"] = df["Y_t"].shift(1)

m = df[["Date", "Y_t", "Y_t_lag1", "Dt"]].dropna().copy()
X = sm.add_constant(m[["Y_t_lag1", "Dt"]])
model = sm.OLS(m["Y_t"], X).fit()
m["Predicted"] = model.predict(X)
m["Residual"] = m["Y_t"] - m["Predicted"]

s = np.sqrt(model.scale)
event = m[m["Dt"] == 1].copy()
event["z"] = event["Residual"] / s
print(event[["Date", "Y_t", "Predicted", "Residual", "z"]].round(3))
print("MAE sự kiện:", round(event["Residual"].abs().mean(), 3),
      "| MAE ngày thường:", round(m[m["Dt"] == 0]["Residual"].abs().mean(), 3))

          Date    Y_t  Predicted  Residual      z
192 2023-10-09  0.630     -0.945     1.575  1.708
325 2024-04-19 -0.876     -0.891     0.015  0.017
564 2025-04-03 -4.840     -0.925    -3.914 -4.245
767 2026-01-26  0.501     -0.901     1.401  1.520
791 2026-03-02  0.040     -0.883     0.923  1.001
MAE sự kiện: 1.566 | MAE ngày thường: 0.645
